# S8 — Advanced pandas

`merge` / `join`, string cleaning, dates, missing values, MultiIndex and categorical dtype.


## 1. Merge and join

```python
left.merge(right, on="key", how="left")
left.join(right.set_index("key"), on="key", how="left")
```

- `merge`: column ↔ column
- `join`: left column ↔ right index
- `indicator=True`: adds the `_merge` column
- `validate="many_to_one"`: the key on the right must be unique


In [1]:
import pandas as pd

employees = pd.DataFrame({
    "employee_id": [101, 102, 103, 104, 105, 106],
    "name": ["Ada", "Bora", "Cem", "Deniz", "Ece", "Fırat"],
    "department_id": [10, 20, 10, 20, 30, 50],
    "salary": [72000, 85000, 68000, 92000, 79000, 98000]
})

departments = pd.DataFrame({
    "department_id": [10, 20, 30, 40],
    "department_name": ["ML", "Backend", "Data", "Security"],
    "budget": [500000, 750000, 420000, 350000]
})

### Quick examples


In [2]:
employees.merge(departments,on="department_id") #how = "inner" (default)

,employee_id,name,department_id,salary,department_name,budget
0,101,Ada,10,72000,ML,500000
1,102,Bora,20,85000,Backend,750000
2,103,Cem,10,68000,ML,500000
3,104,Deniz,20,92000,Backend,750000
4,105,Ece,30,79000,Data,420000


In [3]:
employees.merge(departments,on="department_id",how = "left")

,employee_id,name,department_id,salary,department_name,budget
0,101,Ada,10,72000,ML,500000.0
1,102,Bora,20,85000,Backend,750000.0
2,103,Cem,10,68000,ML,500000.0
3,104,Deniz,20,92000,Backend,750000.0
4,105,Ece,30,79000,Data,420000.0
5,106,Fırat,50,98000,NaN,NaN


In [4]:
employees.merge(departments,on="department_id",how = "right")

,employee_id,name,department_id,salary,department_name,budget
0,101.0,Ada,10,72000.0,ML,500000
1,103.0,Cem,10,68000.0,ML,500000
2,102.0,Bora,20,85000.0,Backend,750000
3,104.0,Deniz,20,92000.0,Backend,750000
4,105.0,Ece,30,79000.0,Data,420000
5,NaN,NaN,40,NaN,Security,350000


In [5]:
employees.merge(departments,on="department_id",how = "outer")

,employee_id,name,department_id,salary,department_name,budget
0,101.0,Ada,10,72000.0,ML,500000.0
1,103.0,Cem,10,68000.0,ML,500000.0
2,102.0,Bora,20,85000.0,Backend,750000.0
3,104.0,Deniz,20,92000.0,Backend,750000.0
4,105.0,Ece,30,79000.0,Data,420000.0
5,NaN,NaN,40,NaN,Security,350000.0
6,106.0,Fırat,50,98000.0,NaN,NaN


In [6]:
employees.merge(departments,on="department_id",how="outer",indicator=True)

,employee_id,name,department_id,salary,department_name,budget,_merge
0,101.0,Ada,10,72000.0,ML,500000.0,both
1,103.0,Cem,10,68000.0,ML,500000.0,both
2,102.0,Bora,20,85000.0,Backend,750000.0,both
3,104.0,Deniz,20,92000.0,Backend,750000.0,both
4,105.0,Ece,30,79000.0,Data,420000.0,both
5,NaN,NaN,40,NaN,Security,350000.0,right_only
6,106.0,Fırat,50,98000.0,NaN,NaN,left_only


In [7]:
merged = employees.merge(departments,on="department_id",how="outer",indicator=True,validate="many_to_one")

In [8]:
merged

,employee_id,name,department_id,salary,department_name,budget,_merge
0,101.0,Ada,10,72000.0,ML,500000.0,both
1,103.0,Cem,10,68000.0,ML,500000.0,both
2,102.0,Bora,20,85000.0,Backend,750000.0,both
3,104.0,Deniz,20,92000.0,Backend,750000.0,both
4,105.0,Ece,30,79000.0,Data,420000.0,both
5,NaN,NaN,40,NaN,Security,350000.0,right_only
6,106.0,Fırat,50,98000.0,NaN,NaN,left_only


In [9]:
merged["_merge"].value_counts()

_merge
both          5
left_only     1
right_only    1
Name: count, dtype: int64

In [10]:
merged[merged["_merge"] != "both"]

,employee_id,name,department_id,salary,department_name,budget,_merge
5,NaN,NaN,40,NaN,Security,350000.0,right_only
6,106.0,Fırat,50,98000.0,NaN,NaN,left_only


In [11]:
employees.join(
    departments.set_index("department_id"),
    on="department_id",
    how="left")

,employee_id,name,department_id,salary,department_name,budget
0,101,Ada,10,72000,ML,500000.0
1,102,Bora,20,85000,Backend,750000.0
2,103,Cem,10,68000,ML,500000.0
3,104,Deniz,20,92000,Backend,750000.0
4,105,Ece,30,79000,Data,420000.0
5,106,Fırat,50,98000,NaN,NaN


### Exercise 1 — ★☆☆

Merge `employees` and `departments` on `department_id` with an inner join.


In [12]:
employees.merge(departments,on="department_id",how="inner")

,employee_id,name,department_id,salary,department_name,budget
0,101,Ada,10,72000,ML,500000
1,102,Bora,20,85000,Backend,750000
2,103,Cem,10,68000,ML,500000
3,104,Deniz,20,92000,Backend,750000
4,105,Ece,30,79000,Data,420000


### Exercise 2 — ★☆☆

Keep all employees with a left merge. Check the unmatched values for Fırat.


In [13]:
employees.merge(departments,on="department_id",how="left")

,employee_id,name,department_id,salary,department_name,budget
0,101,Ada,10,72000,ML,500000.0
1,102,Bora,20,85000,Backend,750000.0
2,103,Cem,10,68000,ML,500000.0
3,104,Deniz,20,92000,Backend,750000.0
4,105,Ece,30,79000,Data,420000.0
5,106,Fırat,50,98000,NaN,NaN


### Exercise 3 — ★★☆

Use an outer merge with `indicator=True`. Then show only the rows where `_merge != "both"`.


In [14]:
merged = employees.merge(departments,on="department_id",how="outer",indicator=True)

In [15]:
merged

,employee_id,name,department_id,salary,department_name,budget,_merge
0,101.0,Ada,10,72000.0,ML,500000.0,both
1,103.0,Cem,10,68000.0,ML,500000.0,both
2,102.0,Bora,20,85000.0,Backend,750000.0,both
3,104.0,Deniz,20,92000.0,Backend,750000.0,both
4,105.0,Ece,30,79000.0,Data,420000.0,both
5,NaN,NaN,40,NaN,Security,350000.0,right_only
6,106.0,Fırat,50,98000.0,NaN,NaN,left_only


In [16]:
merged[merged["_merge"] != "both"]

,employee_id,name,department_id,salary,department_name,budget,_merge
5,NaN,NaN,40,NaN,Security,350000.0,right_only
6,106.0,Fırat,50,98000.0,NaN,NaN,left_only


### Exercise 4 — ★★☆

Validate the employee-to-department relationship as `many_to_one`.


In [17]:
employees.merge(departments,on="department_id",validate="many_to_one")

,employee_id,name,department_id,salary,department_name,budget
0,101,Ada,10,72000,ML,500000
1,102,Bora,20,85000,Backend,750000
2,103,Cem,10,68000,ML,500000
3,104,Deniz,20,92000,Backend,750000
4,105,Ece,30,79000,Data,420000


### Exercise 5 — ★★☆

Create `departments_bad` with a duplicated `department_id`.

- Check that `validate="many_to_one"` catches the duplicate.
- Merge again without `validate` and inspect the duplicated employees.


In [18]:
departments_bad = pd.DataFrame({
    "department_id": [10, 20, 20, 30],
    "department_name": [
        "ML",
        "Backend",
        "Backend Duplicate",
        "Data"
    ],
    "budget": [
        500000,
        750000,
        800000,
        420000
    ]
})

In [19]:
try:
    employees.merge(departments_bad,on="department_id",validate="many_to_one")
except pd.errors.MergeError as error:
    print(error)


Merge keys are not unique in right dataset; not a many-to-one merge


In [20]:
employees.merge(departments_bad,on="department_id")

,employee_id,name,department_id,salary,department_name,budget
0,101,Ada,10,72000,ML,500000
1,102,Bora,20,85000,Backend,750000
2,102,Bora,20,85000,Backend Duplicate,800000
3,103,Cem,10,68000,ML,500000
4,104,Deniz,20,92000,Backend,750000
5,104,Deniz,20,92000,Backend Duplicate,800000
6,105,Ece,30,79000,Data,420000


### Exercise 6 — ★★☆

Set `department_id` as the index of `departments` and reproduce the left merge with `.join()`.


In [21]:
employees.join(departments.set_index("department_id"),on = "department_id",how = "left")

,employee_id,name,department_id,salary,department_name,budget
0,101,Ada,10,72000,ML,500000.0
1,102,Bora,20,85000,Backend,750000.0
2,103,Cem,10,68000,ML,500000.0
3,104,Deniz,20,92000,Backend,750000.0
4,105,Ece,30,79000,Data,420000.0
5,106,Fırat,50,98000,NaN,NaN


## 2. String operations and regex

```python
series.str.strip()
series.str.lower()
series.str.contains(pattern, na=False)
series.str.replace(pattern, replacement, regex=True)
series.str.extract(pattern, expand=False)
```

- `na=False`: missing values become `False` in a boolean mask
- `r"\D"`: any non-digit character
- `(...)`: the part captured by `.str.extract()`


In [22]:
contacts = pd.DataFrame({
    "employee_id": [101, 102, 103, 104, 105, 106],
    "name": [
        " Ada Yılmaz ",
        "Bora Demir",
        "Cem Kaya",
        "Deniz Aksoy",
        "Ece Şahin ",
        "Fırat Arslan"
    ],
    "email": [
        "ADA.YILMAZ@COMPANY.COM",
        "bora.demir@company.com",
        "cem.kaya@gmail.com",
        "deniz_aksoy@company.com",
        "ece.sahin@company",
        None
    ],
    "phone": [
        "+90 (532) 111-2233",
        "0533 222 33 44",
        "534-333-4455",
        "+90 535 444 55 66",
        "536.555.6677",
        None
    ]
})

contacts

,employee_id,name,email,phone
0,101,Ada Yılmaz,ADA.YILMAZ@COMPANY.COM,+90 (532) 111-2233
1,102,Bora Demir,bora.demir@company.com,0533 222 33 44
2,103,Cem Kaya,cem.kaya@gmail.com,534-333-4455
3,104,Deniz Aksoy,deniz_aksoy@company.com,+90 535 444 55 66
4,105,Ece Şahin,ece.sahin@company,536.555.6677
5,106,Fırat Arslan,None,None


### Exercise 7 — ★☆☆

Remove the leading and trailing spaces from the `name` column.


In [23]:
contacts["name"] = contacts["name"].str.strip()

### Exercise 8 — ★☆☆

Convert all email addresses to lowercase.


In [24]:
contacts["email"] =  contacts["email"].str.lower()

In [25]:
contacts

,employee_id,name,email,phone
0,101,Ada Yılmaz,ada.yilmaz@company.com,+90 (532) 111-2233
1,102,Bora Demir,bora.demir@company.com,0533 222 33 44
2,103,Cem Kaya,cem.kaya@gmail.com,534-333-4455
3,104,Deniz Aksoy,deniz_aksoy@company.com,+90 535 444 55 66
4,105,Ece Şahin,ece.sahin@company,536.555.6677
5,106,Fırat Arslan,None,None


### Exercise 9 — ★☆☆

Select employees whose email contains the plain text `@company.com`. Use `regex=False` and `na=False`.


In [26]:
contacts[contacts["email"].str.contains("@company.com",na=False,regex = False)]

,employee_id,name,email,phone
0,101,Ada Yılmaz,ada.yilmaz@company.com,+90 (532) 111-2233
1,102,Bora Demir,bora.demir@company.com,0533 222 33 44
3,104,Deniz Aksoy,deniz_aksoy@company.com,+90 535 444 55 66


### Exercise 10 — ★★☆

Remove every non-digit character from `phone` and store the result in `phone_digits`.


In [27]:
contacts["phone_digits"] = contacts["phone"].str.replace(r"\D", "",regex = True)

In [28]:
contacts

,employee_id,name,email,phone,phone_digits
0,101,Ada Yılmaz,ada.yilmaz@company.com,+90 (532) 111-2233,905321112233
1,102,Bora Demir,bora.demir@company.com,0533 222 33 44,05332223344
2,103,Cem Kaya,cem.kaya@gmail.com,534-333-4455,5343334455
3,104,Deniz Aksoy,deniz_aksoy@company.com,+90 535 444 55 66,905354445566
4,105,Ece Şahin,ece.sahin@company,536.555.6677,5365556677
5,106,Fırat Arslan,None,None,None


### Exercise 11 — ★★☆

Extract the email domain into `email_domain`. Return a Series with `expand=False`.


In [29]:
contacts["email_domain"] = contacts["email"].str.extract(r"@(.+)$",expand = False)

In [30]:
contacts

,employee_id,name,email,phone,phone_digits,email_domain
0,101,Ada Yılmaz,ada.yilmaz@company.com,+90 (532) 111-2233,905321112233,company.com
1,102,Bora Demir,bora.demir@company.com,0533 222 33 44,05332223344,company.com
2,103,Cem Kaya,cem.kaya@gmail.com,534-333-4455,5343334455,gmail.com
3,104,Deniz Aksoy,deniz_aksoy@company.com,+90 535 444 55 66,905354445566,company.com
4,105,Ece Şahin,ece.sahin@company,536.555.6677,5365556677,company
5,106,Fırat Arslan,None,None,None,None


### Exercise 12 — ★★☆

Create a `valid_email` column with the given regex. Then show invalid or missing email addresses.


In [31]:
email_pattern = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

In [32]:
contacts["valid_email"] = (contacts["email"].str.lower().str.contains(email_pattern,regex=True,na=False))

In [33]:
contacts[~contacts["valid_email"]]

,employee_id,name,email,phone,phone_digits,email_domain,valid_email
4,105,Ece Şahin,ece.sahin@company,536.555.6677,5365556677,company,False
5,106,Fırat Arslan,None,None,None,None,False


## 3. Datetime

```python
pd.to_datetime(series, errors="coerce")
series.dt.year
pd.date_range(start, periods=..., freq="MS")
df.resample("MS", on="date")
```

- `errors="coerce"`: invalid dates become `NaT`
- `.dt`: year, month and day access
- `resample`: time-based `groupby`


In [34]:
sales = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007],
    "order_date": [
        "2026-01-05",
        "2026-01-18",
        "2026-02-03",
        "2026-02-21",
        "2026-03-02",
        "invalid-date",
        None
    ],
    "amount": [1200, 850, 2100, 1600, 900, 500, 750]
})

sales

,order_id,order_date,amount
0,1001,2026-01-05,1200
1,1002,2026-01-18,850
2,1003,2026-02-03,2100
3,1004,2026-02-21,1600
4,1005,2026-03-02,900
5,1006,invalid-date,500
6,1007,None,750


### Exercise 13 — ★☆☆

Convert `order_date` to datetime. Invalid values should become `NaT`.


In [35]:
sales["order_date"] = pd.to_datetime(sales["order_date"],errors = "coerce")

In [36]:
sales

,order_id,order_date,amount
0,1001,2026-01-05,1200
1,1002,2026-01-18,850
2,1003,2026-02-03,2100
3,1004,2026-02-21,1600
4,1005,2026-03-02,900
5,1006,NaT,500
6,1007,NaT,750


### Exercise 14 — ★☆☆

Create `order_year`, `order_month` and `order_day` with the `.dt` accessor.


In [37]:
sales["order_year"] = sales["order_date"].dt.year
sales["order_month"] = sales["order_date"].dt.month
sales["order_day"] = sales["order_date"].dt.day

In [38]:
sales

,order_id,order_date,amount,order_year,order_month,order_day
0,1001,2026-01-05,1200,2026.0,1.0,5.0
1,1002,2026-01-18,850,2026.0,1.0,18.0
2,1003,2026-02-03,2100,2026.0,2.0,3.0
3,1004,2026-02-21,1600,2026.0,2.0,21.0
4,1005,2026-03-02,900,2026.0,3.0,2.0
5,1006,NaT,500,NaN,NaN,NaN
6,1007,NaT,750,NaN,NaN,NaN


### Exercise 15 — ★☆☆

Select the February 2026 orders.


In [39]:
sales[sales["order_month"] == 2]

,order_id,order_date,amount,order_year,order_month,order_day
2,1003,2026-02-03,2100,2026.0,2.0,3.0
3,1004,2026-02-21,1600,2026.0,2.0,21.0


### Exercise 16 — ★☆☆

Create six month-start dates beginning from `2026-01-01`.


In [40]:
pd.date_range("2026-01-01",periods = 6, freq = "MS")

DatetimeIndex(['2026-01-01', '2026-02-01', '2026-03-01', '2026-04-01',
               '2026-05-01', '2026-06-01'],
              dtype='datetime64[ns]', freq='MS')

### Exercise 17 — ★★☆

Resample the sales monthly and calculate total sales for each month.


In [41]:
sales.resample("MS",on = "order_date")["amount"].sum()

order_date
2026-01-01    2050
2026-02-01    3700
2026-03-01     900
Name: amount, dtype: int64

### Exercise 18 — ★★☆

Create a monthly summary with `total_sales`, `average_sale` and `order_count`.


In [42]:
sales.resample("MS",on = "order_date").agg(total_sales = ("amount","sum"),
                                           average_sale = ("amount","mean"),
                                           order_count = ("order_id","count"))


,total_sales,average_sale,order_count
order_date,,,
2026-01-01,2050,1025.0,2
2026-02-01,3700,1850.0,2
2026-03-01,900,900.0,1


## 4. Missing values

```python
df.isna().sum()
series.fillna(value)
df.dropna(subset=[...], how="all")
```

- `how="all"`: drop the row only if all selected columns are missing
- `how="any"`: drop the row if at least one selected column is missing
- `fillna()` does not change the original object unless the result is assigned


In [43]:
projects = pd.DataFrame({
    "project_id": [1, 2, 3, 4, 5, 6],
    "department": ["ML", "Backend", None, "Data", "ML", None],
    "budget": [120000, None, 90000, 150000, None, 110000],
    "manager": ["Ada", "Bora", None, "Deniz", "Ece", None]
})

### Exercise 19 — ★☆☆

Count the missing values in each column.


In [44]:
projects.isna().sum()

project_id    0
department    2
budget        2
manager       2
dtype: int64

### Exercise 20 — ★☆☆

Select the rows where `department` is missing.


In [45]:
projects[projects["department"].isna()]

,project_id,department,budget,manager
2,3,None,90000.0,None
5,6,None,110000.0,None


### Exercise 21 — ★☆☆

Copy the DataFrame and fill missing departments with `"Unknown"`.


In [46]:
projectsWithDepartmentsFilled = projects.copy()

In [47]:
projectsWithDepartmentsFilled["department"] = projects["department"].fillna("Unknown")


In [48]:
projectsWithDepartmentsFilled

,project_id,department,budget,manager
0,1,ML,120000.0,Ada
1,2,Backend,NaN,Bora
2,3,Unknown,90000.0,None
3,4,Data,150000.0,Deniz
4,5,ML,NaN,Ece
5,6,Unknown,110000.0,None


### Exercise 22 — ★★☆

Fill missing budgets with the mean of the available budgets.


In [49]:
projectsWithDepartmentsFilled["budget"] = (
    projectsWithDepartmentsFilled["budget"]
    .fillna(projectsWithDepartmentsFilled["budget"].mean())
)


In [50]:
projectsWithDepartmentsFilled

,project_id,department,budget,manager
0,1,ML,120000.0,Ada
1,2,Backend,117500.0,Bora
2,3,Unknown,90000.0,None
3,4,Data,150000.0,Deniz
4,5,ML,117500.0,Ece
5,6,Unknown,110000.0,None


### Exercise 23 — ★★☆

Drop rows where both `department` and `manager` are missing.


In [51]:
projects.dropna(subset = ["department","manager"], how= "all")

,project_id,department,budget,manager
0,1,ML,120000.0,Ada
1,2,Backend,NaN,Bora
3,4,Data,150000.0,Deniz
4,5,ML,NaN,Ece


## 5. MultiIndex and categorical dtype

```python
df.groupby(["column1", "column2"])
summary.loc["ML"]
summary.loc[("Backend", 2026)]
summary.reset_index()
series.astype("category")
```

- Multiple group columns create a hierarchical index by default
- `.reset_index()` turns index levels back into normal columns
- `category` is useful for columns with a small number of repeated values


In [52]:
performance = pd.DataFrame({
    "department": [
        "ML", "ML", "ML",
        "Backend", "Backend", "Backend",
        "Data", "Data"
    ],
    "year": [
        2025, 2025, 2026,
        2025, 2026, 2026,
        2025, 2026
    ],
    "employee": [
        "Ada", "Cem", "Ada",
        "Bora", "Bora", "Deniz",
        "Ece", "Ece"
    ],
    "score": [88, 82, 93, 85, 89, 91, 84, 90],
    "completed_projects": [4, 3, 6, 5, 6, 7, 4, 5]
})

performance

,department,year,employee,score,completed_projects
0,ML,2025,Ada,88,4
1,ML,2025,Cem,82,3
2,ML,2026,Ada,93,6
3,Backend,2025,Bora,85,5
4,Backend,2026,Bora,89,6
5,Backend,2026,Deniz,91,7
6,Data,2025,Ece,84,4
7,Data,2026,Ece,90,5


### Exercise 24 — ★☆☆

Group by `department` and `year`. Calculate `average_score` and `total_projects`, then inspect the index.


In [53]:
summary = performance.groupby(["department","year"]).agg(
    average_score = ("score","mean"),
    total_projects = ("completed_projects","sum"))

In [54]:
summary

average_score  total_projects
department year                               
Backend    2025           85.0               5
           2026           90.0              13
Data       2025           84.0               4
           2026           90.0               5
ML         2025           85.0               7
           2026           93.0               6

In [55]:
summary.index

MultiIndex([('Backend', 2025),
            ('Backend', 2026),
            (   'Data', 2025),
            (   'Data', 2026),
            (     'ML', 2025),
            (     'ML', 2026)],
           names=['department', 'year'])

### Exercise 25 — ★☆☆

Select only the `ML` department from the MultiIndex result.


In [56]:
summary.loc["ML"]

,average_score,total_projects
year,,
2025,85.0,7
2026,93.0,6


### Exercise 26 — ★★☆

Select the row for `Backend`, `2026`.


In [57]:
summary.loc[("Backend",2026)]


average_score     90.0
total_projects    13.0
Name: (Backend, 2026), dtype: float64

### Exercise 27 — ★☆☆

Flatten the result with `.reset_index()`.


In [58]:
flat_summary = summary.reset_index()

In [59]:
flat_summary

,department,year,average_score,total_projects
0,Backend,2025,85.0,5
1,Backend,2026,90.0,13
2,Data,2025,84.0,4
3,Data,2026,90.0,5
4,ML,2025,85.0,7
5,ML,2026,93.0,6


### Exercise 28 — ★★☆

Convert `department` to `category`, then inspect `.cat.categories` and `.cat.codes`.


In [60]:
performance["department"] = (performance["department"].astype("category"))

In [61]:
performance["department"]

0         ML
1         ML
2         ML
3    Backend
4    Backend
5    Backend
6       Data
7       Data
Name: department, dtype: category
Categories (3, object): ['Backend', 'Data', 'ML']

In [62]:
performance["department"].cat.categories

Index(['Backend', 'Data', 'ML'], dtype='object')

In [63]:
performance["department"].cat.codes


0    2
1    2
2    2
3    0
4    0
5    0
6    1
7    1
dtype: int8

## Notes

- Prefer `merge()` when both keys are regular columns.
- Use `join()` when the key on the right is already an index.
- Use `validate=` when a merge is expected to have a specific relationship.
- Use `category` for repeated labels, not mostly unique text such as emails or IDs.
